# Fixtures Analysis

Loads `fixtures` and `fixtures_enhanced` written by `fixtures_etl.py` and explores standings and scoring.

In [ ]:
import duckdb, glob, pandas as pd, numpy as np, plotly.express as px

In [ ]:
db_candidates = sorted(glob.glob('../../data/mydb2024-25*.duckdb'))
db_path = db_candidates[-1] if db_candidates else '../data/mydb.duckdb'
print(f'Using DuckDB database: {db_path}')
con = duckdb.connect(db_path)

In [ ]:
df_fixtures = con.execute('SELECT * FROM fixtures').df()
df_fixtures.head()

In [ ]:
# Enhanced table (may not exist on first run)
try:
    df_fx_enh = con.execute('SELECT * FROM fixtures_enhanced').df()
except Exception:
    df_fx_enh = pd.DataFrame()
df_fx_enh.head() if not df_fx_enh.empty else 'fixtures_enhanced not available' 

In [ ]:
# Parse times and scores
for col in ['score_home','score_away']:
    if col in df_fixtures.columns:
        df_fixtures[col] = pd.to_numeric(df_fixtures[col], errors='coerce')
if 'startTimeUTC' in df_fixtures.columns:
    df_fixtures['startTimeUTC'] = pd.to_datetime(df_fixtures['startTimeUTC'], errors='coerce')
df_fixtures['goals_total'] = df_fixtures.get('score_home',0) + df_fixtures.get('score_away',0)
df_fixtures[['score_home','score_away','goals_total']].describe()

In [ ]:
# Goals distribution
px.histogram(df_fixtures, x='goals_total', nbins=30, title='Total Goals per Match')

In [ ]:
# Top scoring teams (home+away)
home = df_fixtures.groupby('name_team_home')['score_home'].sum().rename('goals_home') if 'name_team_home' in df_fixtures.columns else pd.Series(dtype=float)
away = df_fixtures.groupby('name_team_away')['score_away'].sum().rename('goals_away') if 'name_team_away' in df_fixtures.columns else pd.Series(dtype=float)
df_goals = pd.concat([home, away], axis=1).fillna(0)
df_goals['goals_total'] = df_goals.sum(axis=1)
px.bar(df_goals.sort_values('goals_total', ascending=False).reset_index().head(20), x='index', y='goals_total', title='Top Teams by Goals (Total)')

In [ ]:
# Standings snapshot from enhanced table (if available)
if not df_fx_enh.empty:
    cols = [c for c in ['wins_home','draws_home','losses_home','wins_away','draws_away','losses_away'] if c in df_fx_enh.columns]
    display(df_fx_enh[cols].describe())
else:
    print('fixtures_enhanced not available yet.')